In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import StrMethodFormatter

# 1. Filter for Data Analysts and remove rows without salary data
df_DA = df[df['job_title_short'] == 'Data Analyst'].copy()
df_DA_salary = df_DA.dropna(subset=['salary_year_avg', 'job_skills']).copy()

# 2. Clean and Explode the skills
df_DA_salary['job_skills'] = df_DA_salary['job_skills'].str.replace(r"[\[\]']", "", regex=True).str.split(", ")
df_DA_exploded = df_DA_salary.explode('job_skills')

# 3. Calculate both the Median Salary and the Count of each skill
# We use Median instead of Mean because salaries are often skewed by extreme outliers
skill_stats = df_DA_exploded.groupby('job_skills').agg(
    median_salary=('salary_year_avg', 'median'),
    skill_count=('job_skills', 'count')
).reset_index()

# 4. THE STATISTICAL FIX: Only keep skills that appear in at least 5% of the salary postings
# (You can adjust this number up or down depending on your dataset)
min_count = len(df_DA_salary) * 0.05 
top_paying_skills = skill_stats[skill_stats['skill_count'] >= min_count]

# 5. Sort by highest salary and grab the Top 10
top_paying_skills = top_paying_skills.sort_values(by='median_salary', ascending=False).head(10)

# ---------------------------------------------------------
# THE VISUALIZATION
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=top_paying_skills, x='median_salary', y='job_skills', palette='viridis')

plt.title('Top 10 Highest Paying Skills for Data Analysts', fontsize=15)
plt.xlabel('Median Yearly Salary ($)', fontsize=12)
plt.ylabel('')

# Format X-axis as Currency (e.g., $100,000)
ax.xaxis.set_major_formatter(StrMethodFormatter('${x:,.0f}'))

# Add salary labels to the end of each bar
for i, p in enumerate(ax.patches):
    ax.annotate(f'${int(p.get_width()):,}', 
                (p.get_width(), p.get_y() + p.get_height() / 2), 
                ha='left', va='center', xytext=(5, 0), textcoords='offset points', fontweight='bold')

# Clean up borders
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

NameError: name 'df' is not defined